# Bonus: when does XCiT stop trusting the signal?

This five-minute experiment asks one simple question: **what happens to a classifier as its input disappears into noise?**

We'll create a complex tone, reuse the same noise realization at several SNRs, and run every version through the official 57-class XCiT checkpoint from TorchSig Models v1.0.0. Then we'll compare what we can see in a spectrogram with what the model reports through its predicted class, confidence, and entropy.

## 1. Set up the runtime

Run this cell once in Colab or a local notebook environment. It installs only missing packages. The checkpoint itself is downloaded in the following section and cached beside the notebook.

In [ ]:
import importlib.util
import subprocess
import sys

requirements = []
if importlib.util.find_spec('torchsig_models') is None:
    requirements.append('git+https://github.com/TorchDSP/torchsig-models.git@v1.0.0')
if importlib.util.find_spec('matplotlib') is None:
    requirements.append('matplotlib>=3.7')
if requirements:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *requirements])
    print('Installation complete. Restart the runtime if an import still fails.')
else:
    print('All dependencies are ready.')

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch

from torchsig_models.models import XCiTClassifier

CHECKPOINT = Path('xcit_narrowband_v1.0.0.ckpt')
CHECKPOINT_URL = (
    'https://github.com/TorchDSP/torchsig-models/releases/download/'
    'v1.0.0/xcit_narrowband_v1.0.0.ckpt'
)
if not CHECKPOINT.exists():
    print('Downloading the official XCiT checkpoint...')
    urlretrieve(CHECKPOINT_URL, CHECKPOINT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Checkpoint: {CHECKPOINT.resolve()}')
print(f'Device: {DEVICE}')

## 2. Hold the signal still and turn up the noise

The reference is a unit-power complex tone at 15% of the sample rate. We generate one unit-power complex noise vector and scale that **same vector** at every SNR. Holding the signal and noise realization fixed makes this a clean comparison: only noise amplitude changes.

SNR is a power ratio, so noise amplitude is scaled by `10 ** (-SNR / 20)`. The resulting array has shape `[SNR levels, samples]`.

In [ ]:
SEED = 2026
NUM_SAMPLES = 4096
SNR_DB = np.array([-10, -5, 0, 5, 10, 20, 30])
rng = np.random.default_rng(SEED)

sample_index = np.arange(NUM_SAMPLES)
clean_iq = np.exp(2j * np.pi * 0.15 * sample_index).astype(np.complex64)
noise = (rng.standard_normal(NUM_SAMPLES) + 1j * rng.standard_normal(NUM_SAMPLES)).astype(np.complex64)
noise /= np.sqrt(np.mean(np.abs(noise) ** 2))

noisy_iq = np.stack([
    clean_iq + noise * 10 ** (-snr / 20)
    for snr in SNR_DB
]).astype(np.complex64)
print('Sweep shape:', noisy_iq.shape)

In [ ]:
selected = [0, 2, 4, 6]  # -10, 0, 10, and 30 dB
fig, axes = plt.subplots(1, len(selected), figsize=(14, 3.2), constrained_layout=True)
for axis, index in zip(axes, selected):
    axis.specgram(noisy_iq[index], NFFT=256, Fs=1.0, noverlap=192, cmap='magma')
    axis.set_title(f'{SNR_DB[index]} dB')
    axis.set_xlabel('Normalized time')
axes[0].set_ylabel('Normalized frequency')
fig.suptitle('The tone becomes visible as SNR increases')
plt.show()

## 3. Ask the released model

XCiT expects two real channels rather than a complex NumPy dtype, so we stack I and Q to form `[batch, 2, samples]`. The checkpoint produces raw logits for 57 historical TorchSig classes. Softmax turns those logits into a distribution we can inspect.

The class-name order below is intentionally fixed to TorchSig 2.1.1, the version used to train the release checkpoint. Substituting the class registry from a newer TorchSig installation can give the right index the wrong name.

In [ ]:
CLASS_NAMES_V211 = [
    'tone', 'ofdm-64', 'ofdm-72', 'ofdm-128', 'ofdm-180', 'ofdm-256',
    'ofdm-300', 'ofdm-512', 'ofdm-600', 'ofdm-900', 'ofdm-1024',
    'ofdm-1200', 'ofdm-2048', 'lfm-data', 'lfm-radar', '2fsk', '4fsk',
    '8fsk', '16fsk', '2gfsk', '4gfsk', '8gfsk', '16gfsk', '2msk',
    '4msk', '8msk', '16msk', '2gmsk', '4gmsk', '8gmsk', '16gmsk',
    'fm', 'ook', 'bpsk', 'qpsk', '8psk', '16psk', '32psk', '64psk',
    '4ask', '8ask', '16ask', '32ask', '64ask', '16qam', '32qam',
    '64qam', '256qam', '1024qam', '32qam_cross', '128qam_cross',
    '512qam_cross', 'chirpss', 'am-dsb', 'am-dsb-sc', 'am-usb', 'am-lsb',
]

model = XCiTClassifier.load_from_checkpoint(CHECKPOINT, map_location=DEVICE)
model.to(DEVICE).eval()
model_input = torch.from_numpy(
    np.stack((noisy_iq.real, noisy_iq.imag), axis=1)
).to(device=DEVICE, dtype=torch.float32)

with torch.inference_mode():
    probabilities = model(model_input).softmax(dim=1).cpu()

confidence, predicted_index = probabilities.max(dim=1)
predicted_names = [CLASS_NAMES_V211[index] for index in predicted_index.tolist()]
for snr, name, score in zip(SNR_DB, predicted_names, confidence):
    print(f'{snr:>3} dB: {name:12s} confidence={score.item():.1%}')

## 4. Confidence is only half the story

The left plot follows the largest softmax value. The right plot shows normalized entropy, which summarizes how spread out the entire distribution is: zero means one class dominates, while one means probability is distributed uniformly across all 57 classes.

A sensible trend is useful evidence that the model responds to signal quality, but softmax confidence is **not automatically calibrated probability**. A model can be confidently wrong, especially when an analytic signal differs from its synthetic training distribution.

In [ ]:
safe_probabilities = probabilities.clamp_min(torch.finfo(probabilities.dtype).tiny)
entropy = -(safe_probabilities * safe_probabilities.log()).sum(dim=1)
normalized_entropy = entropy / np.log(len(CLASS_NAMES_V211))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
axes[0].plot(SNR_DB, confidence.numpy(), marker='o', linewidth=2)
for snr, score, name in zip(SNR_DB, confidence, predicted_names):
    axes[0].annotate(name, (snr, score.item()), xytext=(0, 7), textcoords='offset points', ha='center', fontsize=8)
axes[0].set(title='Top-class confidence', xlabel='SNR (dB)', ylabel='Softmax confidence', ylim=(0, 1.05))
axes[1].plot(SNR_DB, normalized_entropy.numpy(), marker='o', color='#D55E00', linewidth=2)
axes[1].set(title='Uncertainty across all classes', xlabel='SNR (dB)', ylabel='Normalized entropy', ylim=(0, 1.05))
for axis in axes:
    axis.grid(alpha=0.25)
plt.show()

In [ ]:
top_values, top_indices = probabilities.topk(3, dim=1)
fig, axes = plt.subplots(1, len(SNR_DB), figsize=(18, 3.4), sharey=True, constrained_layout=True)
for axis, snr, values, indices in zip(axes, SNR_DB, top_values, top_indices):
    names = [CLASS_NAMES_V211[index] for index in indices.tolist()]
    axis.barh(names[::-1], values.numpy()[::-1], color='#4472C4')
    axis.set(title=f'{snr} dB', xlim=(0, 1))
    axis.tick_params(axis='y', labelsize=8)
axes[0].set_xlabel('Probability')
fig.suptitle('Top three classes at each SNR')
plt.show()

## 5. The five-minute takeaway

We changed one physical variable—noise power—and watched both the signal representation and model output evolve. That is a compact robustness test, and the pattern generalizes: sweep frequency offset, phase, bandwidth, fading, or interference while holding everything else fixed.

The important habit is to inspect the **whole prediction distribution**, not only the winning label. Confidence, entropy, and competing classes tell a much richer story about where a model begins to struggle.